# Analyse non-supervisée (t-SNE & k-means) — Adult Income (UCI)

Ce notebook prolonge l'AED (`eda.ipynb`) et **réutilise la `PreprocessingPipeline` définie et justifiée dans `preprocessing.ipynb`** (one-hot + **scaling**, indispensable aux distances euclidiennes). Il se place donc *après* le preprocessing dans l'ordre logique du projet :

`eda → preprocessing → unsupervised → modeling → hyperopt`

L'analyse est **non-supervisée** : la cible `income` n'entre jamais dans l'apprentissage, elle ne sert qu'*a posteriori* à colorier et profiler les résultats. Comme l'AED, tout est mené **sur le jeu d'entraînement uniquement**.

Deux objectifs :
- **t-SNE** : projeter l'espace (≈70 dimensions après one-hot) en 2D pour *voir* si les deux classes de revenu forment des régions séparées (lecture qualitative de la difficulté du problème).
- **k-means** : segmenter les individus en groupes homogènes, puis regarder le **taux de `>50K`** par groupe — un écart marqué au ~24 % global révèle des profils socio-économiques discriminants.

Tout passe par `ClusteringAnalyzer`, qui enchaîne la pipeline puis une **PCA intermédiaire** partagée par les deux méthodes (débruitage pour k-means, pré-réduction pour t-SNE).

> ⚠️ Limite : après one-hot l'espace est surtout binaire, la distance euclidienne y est un peu artificielle. La PCA atténue ce point ; les conclusions restent **qualitatives**.


## 0. Préparation de l'environnement


In [ ]:
# Recharge automatique des modules src/ modifiés sans restart du kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from adult_income import (
    DataLoader,
    ClusteringAnalyzer,
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    TARGET,
    POSITIVE_CLASS,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Imports OK")


## 1. Chargement des données

> **Protocole** : comme l'AED, l'analyse non-supervisée est menée **uniquement sur le jeu d'entraînement** (`adult.data`, split officiel UCI). Le jeu de test reste de côté.


In [ ]:
loader = DataLoader()
X, y = loader.load_train()  # train uniquement ; le test reste de côté

print(f"Observations (train) : {X.shape[0]:,}")
print(f"Features             : {X.shape[1]}")


## 2. PCA intermédiaire

On réduit d'abord la matrice scalée par PCA. Cette représentation est **partagée** par k-means (débruitage) et t-SNE (pré-réduction recommandée au-delà de ~50 dimensions). L'éboulis ci-dessous indique combien de composantes suffisent à capter l'essentiel de la variance.

In [ ]:
# Réutilise X, y du train (cf. §1). scaling=True : obligatoire pour les distances.
ca = ClusteringAnalyzer(X, y, scaling=True, random_state=42)
print(f"Matrice préprocessée (one-hot + scaling) : {ca.Z.shape}")

In [ ]:
ca.plot_explained_variance(n_components=30)
plt.show()
ca.explained_variance(30).round(3).head(10)

## 3. k-means — choix du nombre de clusters

On cherche un bon `k` en croisant deux indicateurs : la **méthode du coude** (chute puis aplatissement de l'inertie) et le **score de silhouette** (proche de 1 = clusters bien séparés). k-means passe bien à l'échelle sur ~32k lignes (pas de sous-échantillonnage à l'entraînement) ; seule la silhouette, en O(n²), est estimée sur un échantillon. On utilise `k-means++` et plusieurs `n_init` pour limiter la sensibilité à l'initialisation.

In [ ]:
ca.plot_kmeans_search(k_values=range(2, 11))
plt.show()

## 4. Profilage des clusters

On ajuste le k-means final puis on **profile** chaque groupe sur les variables *brutes* (lisibles) : effectif, **taux de `>50K`**, moyennes numériques et modalité dominante des catégorielles clés. Les groupes dont le taux de hauts revenus s'écarte fortement du ~24 % global correspondent à des profils socio-économiques discriminants.

In [ ]:
# Ajuster k selon le compromis coude/silhouette lu ci-dessus.
ca.fit_kmeans(k=4)
ca.cluster_profile().round(1)

### 4.1. Comparaison k=2 vs k=4

La silhouette est maximale à **k=2** (~0.63, contre ~0.18 dès k=3) : c'est la partition la plus « propre », mais aussi la plus grossière. **k=4** sacrifie un peu de séparation pour des profils plus fins. On affiche les deux profils pour les opposer (puis on rétablit k=4 pour le t-SNE par cluster ci-dessous).

In [ ]:
# k=2 : partition la plus nette (silhouette max) ; k=4 : profils plus fins.
ca.fit_kmeans(k=2)
print("=== k=2 ===")
display(ca.cluster_profile().round(1))

ca.fit_kmeans(k=4)  # on rétablit k=4 pour le t-SNE par cluster ci-dessous
print("=== k=4 ===")
display(ca.cluster_profile().round(1))

**Lecture comparée :**

- **k=2** se contente d'**isoler l'îlot capital-gain** (8 % de la population, ~62 % de `>50K`) du reste. Mais le second cluster regroupe alors 92 % des individus à un taux de ~21 %, proche du ~24 % global : très bien séparé (silhouette élevée) mais **peu informatif**, il ne fait que confirmer ce qu'on savait déjà sur capital-gain.
- **k=4** conserve cet îlot (cluster à 62 %) et **décompose les 92 % restants** en un dégradé socio-économique exploitable : cadres aux longues heures (~43 %), profil plus âgé peu diplômé (~13 %), jeunes célibataires (~3 %). La silhouette est plus faible car ces groupes se touchent dans l'espace, mais l'information est plus riche.

**Conclusion :** k=2 est le « bon » découpage au sens de la métrique, mais pour décrire des profils dans le rapport, **k=4 est plus parlant**. Les deux racontent la même histoire de fond : la variable la plus discriminante n'est pas une frontière nette mais un petit groupe à capital-gain élevé, le reste de la population formant un continuum.

## 5. t-SNE — visualisation 2D

t-SNE projette les données en 2D en préservant les voisinages locaux. Comme il est coûteux sur ~32k lignes, on travaille sur un **échantillon stratifié** par `income` (~6k points, ratio de classes préservé) et on part de la représentation PCA. `perplexity` (~5–50) influence le rendu.

On trace deux versions du **même** nuage :
1. colorié par `income` → les classes se séparent-elles ?
2. colorié par **cluster k-means** → validation croisée : si les clusters coïncident avec des régions cohérentes du t-SNE, ils captent une vraie structure.

> ⚠️ Sur un t-SNE, les **distances entre clusters et leurs tailles ne sont pas fiables** : lecture purement qualitative.

In [ ]:
ca.plot_tsne(color_by="income", sample_size=6000, perplexity=30)
plt.show()

In [ ]:
ca.plot_tsne(color_by="cluster", sample_size=6000, perplexity=30)
plt.show()

## 6. Bilan de l'analyse non-supervisée

**À retenir :**
- Le **profil des clusters k-means** fait ressortir des groupes au taux de `>50K` très contrasté (du quasi 0 % à la majorité de hauts revenus), pilotés par l'âge, le niveau d'éducation, la profession et le statut marital — cohérent avec les associations vues en bivarié (§6 de l'AED).
- Le **t-SNE colorié par `income`** montre des classes qui se chevauchent largement sans frontière nette : le signal existe mais reste diffus, ce qui annonce qu'aucun modèle ne séparera trivialement les revenus (à confronter aux scores du notebook de modélisation).
- La superposition t-SNE / clusters k-means sert de contrôle de cohérence de la segmentation.

**Limites :** méthodes non-supervisées sur un espace majoritairement one-hot (distance euclidienne artificielle, atténuée mais non corrigée par la PCA) ; t-SNE sur échantillon et purement qualitatif. Ces résultats **illustrent** la structure des données, ils ne remplacent pas la modélisation supervisée.